Instalando bibliotecas e dependências

In [35]:
from qiskit import QuantumCircuit
from qiskit import transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
import numpy as np
import qiskit
import math
from math import sqrt
import matplotlib.pyplot as plt
from qiskit.visualization import array_to_latex
from qiskit.circuit.library import RYGate

# Implementação de gen_angles()

A funçãp gen_angles() recebe um vetor de tamanho 2ˆn e o divide em dois subvetores dimensionais de tamanhos 2ˆn-1. Enquanto o subvetor for maior que 1, ele será recursivamente dividido em dois outros subvetores de tamanho igual. Após a última chamada recursiva, os valores dos ângulos são computados em um novo vetor angles, que retorna como um vetor de ângulos normalizados

In [ ]:
def gen_angles(x):
    if len(x) > 1:
        new_x = list(range(len(x)//2))

        for k in range(len(new_x)):
            new_x[k] = sqrt(x[2*k]**2 + x[2*k+1]**2)

        inner_angles = gen_angles(new_x)

        angles = list(range(len(x)//2))
        for k in range(len(angles)):
            if new_x[k] != 0:
                if x[2*k] > 0:
                    angles[k] = 2 * np.arcsin(x[2*k+1] / new_x[k])
                else:
                    angles[k] = 2 * np.pi - 2 * np.arcsin(x[2*k+1] / new_x[k])
            else:
             angles[k] = 0

        angles = inner_angles + angles
        return angles 
    else:
        return []               

In [37]:
target_angles = [sqrt(0.03), sqrt(0.07), sqrt(0.015), sqrt(0.05), sqrt(0.1), sqrt(0.3),
                 sqrt(0.2), sqrt(0.1)]

target_angles_normalized = gen_angles(target_angles)
print(target_angles_normalized)

UnboundLocalError: cannot access local variable 'inner_angles' where it is not associated with a value

# Implementação do gen_circuit

A função `gen_circuit` constrói o circuito quântico de preparação de estados com base nos ângulos calculados pela função `gen_angles`.

k

In [ ]:
def gen_circuit(angles):
    """
    Gera um circuito quântico de preparação de estados com base nos ângulos calculados.
    
    Parâmetros:
    angles (list): Vetor de dimensão N - 1 contendo os ângulos gerados por gen_angles.
    
    Retorna:
    QuantumCircuit: Circuito quântico gerado.
    """

    N = len(angles) + 1
    n = int(math.log2(N))
    circuit = QuantumCircuit(n)
    q = circuit.qubits

    def level(k):
        """Retorna o nível (altura) do índice k na árvore binária de ângulos (0-indexed)."""
        return int(math.log2(k + 1))

    def index(k, j, q, circuit):
        """
        Aplica portas X nos qubits de controle para representar a condição de controle-0
        (círculo branco) em posições adequadas.
        """
        s = k - (2**j - 1)
        for i in range(j):
            # Obtém o bit correspondente a q[i] na representação de s de j bits.
            # O bit de q[i] corresponde ao bit (j - 1 - i) de s (MSB em q[0], LSB em q[j-1]).
            bit = (s >> (j - 1 - i)) & 1
            if bit == 0:
                circuit.x(q[i])

    for k in range(N - 1):
        j = level(k)
        
        # Mapeia os controles com círculos brancos para controle 1 aplicando X
        index(k, j, q, circuit)
        
        # Aplica a rotação controlada (CRy)
        angle = angles[k]
        if j == 0:
            circuit.ry(angle, q[0])
        else:
            controlled_ry = RYGate(angle).control(num_ctrl_qubits=j)
            circuit.append(controlled_ry, [*q[:j], q[j]])
            
        # Desfaz as portas X para restaurar os estados originais dos qubits
        index(k, j, q, circuit)

    return circuit

# Teste e Validação da Função gen_circuit

Vamos instanciar a função `gen_circuit` com os ângulos do exemplo de 8 dimensões (3 qubits) descritos na Seção 2 do artigo para conferir visualmente a estrutura gerada.

In [ ]:
# Ângulos obtidos a partir do exemplo do artigo para N=8 (3 qubits)
test_angles = [1.98, 1.91, 1.43, 1.98, 1.05, 2.09, 1.23]

# Gerar o circuito quântico
qc = gen_circuit(test_angles)

# Desenhar e exibir o circuito
print("Estrutura do Circuito Quântico Gerado (N=8, 3 qubits):")
print(qc.draw(output='text'))

Estrutura do Circuito Quântico Gerado (N=8, 3 qubits):
     ┌──────────┐┌───┐            ┌───┐            ┌───┐            ┌───┐┌───┐»
q_0: ┤ Ry(1.98) ├┤ X ├─────■──────┤ X ├─────■──────┤ X ├─────■──────┤ X ├┤ X ├»
     └──────────┘└───┘┌────┴─────┐└───┘┌────┴─────┐├───┤     │      ├───┤└───┘»
q_1: ─────────────────┤ Ry(1.91) ├─────┤ Ry(1.43) ├┤ X ├─────■──────┤ X ├─────»
                      └──────────┘     └──────────┘└───┘┌────┴─────┐└───┘     »
q_2: ───────────────────────────────────────────────────┤ Ry(1.98) ├──────────»
                                                        └──────────┘          »
«                 ┌───┐                             
«q_0: ─────■──────┤ X ├─────■────────────────■──────
«          │      ├───┤     │      ┌───┐     │      
«q_1: ─────■──────┤ X ├─────■──────┤ X ├─────■──────
«     ┌────┴─────┐└───┘┌────┴─────┐└───┘┌────┴─────┐
«q_2: ┤ Ry(1.05) ├─────┤ Ry(2.09) ├─────┤ Ry(1.23) ├
«     └──────────┘     └──────────┘     └──────────┘
